In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd

import xarray as xr
import rioxarray
from geocube.api.core import make_geocube
from rasterio.enums import Resampling

import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.colors as mcolors
from matplotlib.colors import ListedColormap, rgb2hex, hex2color, SymLogNorm, Normalize, LogNorm
from matplotlib.patches import Patch as mpatch
from matplotlib.lines import Line2D
from matplotlib.gridspec import GridSpec
from matplotlib.patches import ConnectionPatch, Rectangle

import seaborn as sns
import ultraplot as uplt
import cartopy.crs as ccrs
from pyproj import CRS
from shapely import box

from tempfile import TemporaryDirectory
import shutil

import scipy.ndimage as ndimage
from skimage import morphology, io, color
from skimage.morphology import footprint_rectangle, disk

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

from scipy.stats import ttest_rel

import pylandstats as pls


path_data = Path("E:/GEODATA/climate_war_trap")
path_sample = Path("E:/GEODATA/HANPP_data/global/PSM/sample_point")
path_nc = Path("E:/GEODATA/HANPP_data/global")


In [ ]:
path_koppen = path_data / "koppen_geiger_tif"
da_koppen = xr.open_dataarray(path_koppen / "1991_2020/koppen_geiger_0p5.tif")\
    .sel(band=1).drop_vars("band")

In [ ]:
da_multi_event_m = xr.open_dataarray(path_data / "climate/multi_events_2000_2023.nc")
# da_multi_event_m

In [ ]:
path_continent = Path("E:/GEODATA/HANPP_data/global/vector/globalmap_revise")
dic_region = {
    "Northern Africa and Western Asia": "N. Africa W. Asia",
    "Sub-Saharan Africa": "Sub-Saharan Africa",
    "Central Asia and Russian Federation": "C. Asia",
    "Eastern Asia": "E. Asia",
    "Southern Asia": "S. Asia",
    "Southeastern Asia": "S. Asia",
    "Northern America": "N. America",
    "Latin America and the Caribbean": "Latin America",
    "Western Europe": "W. Europe",
    "Eastern and South-Eastern Europe": "E. Europe",
    "Oceania and Australia": "Oceania",
}
gdf_world = gpd.read_file(path_continent / "map.shp").clip(box(-180, -60, 180, 85))
gdf_world["regi_short"] = gdf_world["regi_pnas"].map(dic_region)
# gdf_world.plot(column="regi_short", cmap="Set3", edgecolor="black", linewidth=0.1, legend=True)

In [ ]:
class Data1year():
    def __init__(self, year):
        self.year = year
        self.da_event = da_multi_event_m.sel(year=self.year)
        self.gdf_conflict = gpd.read_file(path_sample / f"conflict_{self.year}.shp")\
            .assign(year=self.year)\
            .sjoin(gdf_world[["geometry", "regi_pnas", "continent", "subregion", "name_long"]], how="left")\
            .drop(columns=["index_right"])
            
        self.gdf_non_conflict = gpd.read_file(path_sample / f"non_conflict_{self.year}.shp")\
            .assign(year=self.year)\
            .sjoin(gdf_world[["geometry", "regi_pnas", "continent", "subregion", "name_long"]], how="left")\
            .drop(columns=["index_right"])
        
        self.gdf_conflict = self.extract_data(self.gdf_conflict)
        self.gdf_non_conflict = self.extract_data(self.gdf_non_conflict)
        
    def extract_data(self, gdf_):
        return gdf_.assign(
            event=self.da_event.sel(x=xr.DataArray(gdf_.x.values, dims='z'), y=xr.DataArray(gdf_.y.values, dims='z'), method="nearest").values,
            koppen=da_koppen.sel(x=xr.DataArray(gdf_.x.values, dims='z'), y=xr.DataArray(gdf_.y.values, dims='z'), method="nearest").values,
        )
    

In [ ]:
def get_ar_circle(radius):
    radius_clip = {3: 2.54, 4:3.53, 5: 4.49, 6: 5.52, 7: 6.52, 8: 7.49, 9: 8.52, 10: 9.49, 11: 10.52, 12: 11.49, 13: 12.49, 14: 13.51, 15: 14.49, 16: 15.52, 17: 16.46, 18: 17.49, 19: 18.49, 20: 19.47, 21: 20.49, 22: 21.49, 23: 22.49, 24: 23.49, 25: 24.43}[radius]
    gdf_circle = gpd.GeoDataFrame({}, geometry=gpd.GeoDataFrame({}, geometry=gpd.points_from_xy([0], [0])).buffer(radius_clip), crs="epsg:4326")
    da_circle = xr.DataArray(np.ones((radius * 2 + 1, radius * 2 + 1)), coords={"y": np.arange(-radius, radius + 1), "x": np.arange(-radius, radius + 1)})\
        .rio.write_crs("epsg:4326")\
        .rio.clip(gdf_circle.geometry, all_touched=True, drop=False)\
        .fillna(0).astype(np.uint8)
    return da_circle, da_circle.values

In [ ]:
def clip_sample(nc_file, da_circle):
    da_ = xr.open_dataarray(nc_file)
    return xr.where(da_circle==1, da_.sel(x=da_circle.x, y=da_circle.y), np.nan)

In [ ]:
radius = 20
da_circle, ar_circle = get_ar_circle(radius)

In [ ]:
from sklearn.neighbors import BallTree

def get_nearest(src_points, candidates, k_neighbors=1):
    tree = BallTree(candidates, leaf_size=15, metric='haversine')
    distances, indices = tree.query(src_points, k=k_neighbors)
    distances = distances.transpose()
    indices = indices.transpose()
    closest = indices[0]
    closest_dist = distances[0]
    return (closest, closest_dist)

def nearest_neighbor(left_df, right_df, return_dist=False):
    right = right_df.copy().reset_index(drop=True)
    left_radians = np.array(left_df.apply(lambda _df: (_df.x * np.pi / 180, _df.y * np.pi / 180), axis=1).to_list())
    right_radians = np.array(right_df.apply(lambda _df: (_df.x * np.pi / 180, _df.y * np.pi / 180), axis=1).to_list())
    closest, dist = get_nearest(src_points=left_radians, candidates=right_radians)
    closest_points = right.loc[closest]
    closest_points = closest_points.reset_index(drop=True)
    if return_dist:
        earth_radius = 6371000  # meters
        closest_points['distance'] = dist * earth_radius
    return closest_points

## 景观指标

In [ ]:
from tqdm.notebook import tqdm
class Calculate1YearLAMetric:
    def __init__(self, year, radius=20):
        self.year = year
        self.radius = radius
        self.da_circle, self.ar_circle = get_ar_circle(radius)
    
        self.df_conflict = pd.read_csv(path_data / f"sample_Data/koppen_events_conflict_{year}.csv", index_col=0)
        self.df_conflict_lastyear = pd.read_csv(path_data / f"sample_Data/koppen_events_conflict_{year-1}.csv", index_col=0)
        self.df_non_conflict = pd.read_csv(path_data / f"sample_Data/koppen_events_non_conflict_{year}.csv", index_col=0)
        self.LA_result = []
        
        if (path_data / "sample_data/extract_data" / f"{self.year}_LA_metric.csv").exists():
            self.df_LA_result = pd.read_csv(path_data / "sample_data/extract_data" / f"{self.year}_LA_metric.csv", index_col=0)
        else:
            self._load_spatial_data()
            self._cal_LA()

    def _load_spatial_data(self):
        datasets = ["luc_lastyear", "luc_currentyear", ]
        
        for _da in datasets:
            setattr(self, f"da_conflict_{_da}", clip_sample(path_nc / f"PSM25km/conflict_sample/{year}_{_da}.nc", self.da_circle))
            setattr(self, f"da_non_conflict_{_da}", clip_sample(path_nc / f"PSM25km/non_conflict_sample/{year}_{_da}.nc", self.da_circle))
    
    def _cal_LA(self):
        for _conflict in ["conflict", "non_conflict"]:
            for _year in ["currentyear", "lastyear"]:
                _da_luc = getattr(self, f"da_{_conflict}_luc_{_year}")
                for _idx in tqdm(list(_da_luc.idx.values)):
                    _ar_luc = _da_luc.sel(idx=_idx).drop_vars("idx").values
                    
                    luc_types = np.unique(_ar_luc)
                    luc_types = luc_types[~np.isnan(luc_types)]
                    
                    if luc_types.shape[0] == 1:
                        _shannon = 0
                    else:
                        _shannon = pls.Landscape(_ar_luc, res=(1000, 1000)).shannon_diversity_index()
                    
                    self.LA_result.append([_conflict, _year, _idx, _shannon])
                        
        self.df_LA_result = pd.DataFrame(self.LA_result, columns=["conflict", "year", "idx", "shannon"])\
            .pivot(index=["conflict", "idx"], columns="year", values="shannon")\
            .reset_index()
        self.df_LA_result.to_csv(path_data / "sample_data/extract_data" / f"{self.year}_LA_metric.csv")
                

# 提取各类指标

In [ ]:
class Extract1YearData():
    def __init__(self, year, radius=20):
        self.year = year
        self.radius = radius
        self.da_circle, self.ar_circle = get_ar_circle(radius)
    
        self.df_conflict = pd.read_csv(path_data / f"sample_Data/koppen_events_conflict_{year}.csv", index_col=0)
        self.df_conflict_lastyear = pd.read_csv(path_data / f"sample_Data/koppen_events_conflict_{year-1}.csv", index_col=0)
        self.df_non_conflict = pd.read_csv(path_data / f"sample_Data/koppen_events_non_conflict_{year}.csv", index_col=0)

        self.df_conflict_nearest = nearest_neighbor(self.df_conflict, self.df_conflict_lastyear, return_dist=True)
        self.df_non_conflict_nearest = nearest_neighbor(self.df_non_conflict, self.df_conflict_lastyear, return_dist=True)
        
        if (path_data / "sample_data/extract_data" / f"{self.year}_zonal_combined.csv").exists():
            self.df_zonal_combined = pd.read_csv(path_data / "sample_data/extract_data" / f"{self.year}_zonal_combined.csv", index_col=0)
        else:
            self._load_spatial_data()
            self._compute_zonal_stats()

        self._load_LA()
        
        self._seperate_zonal()
        
    def _load_spatial_data(self):
        datasets = ["luc_lastyear", "luc_currentyear", "road_dis", "boundary_dis", "npp_lastyear", "npp_currentyear", "pop", "pop_ly", "fire"]
        
        for _da in datasets:
            setattr(self, f"da_conflict_{_da}", clip_sample(path_nc / f"PSM25km/conflict_sample/{self.year}_{_da}.nc", self.da_circle))
            setattr(self, f"da_non_conflict_{_da}", clip_sample(path_nc / f"PSM25km/non_conflict_sample/{self.year}_{_da}.nc", self.da_circle))
            
    def _compute_zonal_stats(self):
        LAND_COVER_TYPES = {
            'crop': [12, 14],
            'built': [13],
            'forest': [1, 2, 3, 4, 5],
            'shrubland': [6, 7],
            'grass': [10],
            'wetland': [11],
            'barren': [16],
        }

        # 封装通用的统计计算函数
        def calculate_stats(df, da_prefix):
            stats = {
                'pop': getattr(self, f"da_{da_prefix}pop").sum(dim=["x", "y"]).values,
                'pop_ly': getattr(self, f"da_{da_prefix}pop_ly").sum(dim=["x", "y"]).values,
                'fire_area': getattr(self, f"da_{da_prefix}fire").sum(dim=["x", "y"]).values,
                'total_area': (~np.isnan(getattr(self, f"da_{da_prefix}luc_currentyear"))).sum(dim=["x", "y"]).values,
                'total_area_ly': (~np.isnan(getattr(self, f"da_{da_prefix}luc_lastyear"))).sum(dim=["x", "y"]).values,
                'dis2road': getattr(self, f"da_{da_prefix}road_dis").mean(dim=["x", "y"]).values,
                'dis2bound': getattr(self, f"da_{da_prefix}boundary_dis").mean(dim=["x", "y"]).values,
                'npp': getattr(self, f"da_{da_prefix}npp_currentyear").mean(dim=["x", "y"]).values,
                'npp_ly': getattr(self, f"da_{da_prefix}npp_lastyear").mean(dim=["x", "y"]).values,
                'dis2conflict': getattr(self, f"df_{da_prefix}nearest")["distance"].values,
            }

            # 当前年和去年的土地覆盖面积
            current_year_da = getattr(self, f"da_{da_prefix}luc_currentyear")
            last_year_da = getattr(self, f"da_{da_prefix}luc_lastyear")

            for lc_type, codes in LAND_COVER_TYPES.items():
                stats[f'{lc_type}_area'] = current_year_da.isin(codes).sum(dim=["x", "y"]).values
                stats[f'{lc_type}_area_ly'] = last_year_da.isin(codes).sum(dim=["x", "y"]).values

            return df.copy().assign(**stats)

        # 封装比率计算函数
        def calculate_ratios(df):
            df = df.copy()
            for lc_type in LAND_COVER_TYPES:
                df[f'{lc_type}_ratio'] = df[f'{lc_type}_area'] / df.total_area * 100
                df[f'{lc_type}_ratio_ly'] = df[f'{lc_type}_area_ly'] / df.total_area_ly * 100
            return df

        # 封装变化量计算函数
        def calculate_changes(df):
            df = df.copy()
            for lc_type in LAND_COVER_TYPES:
                df[f'{lc_type}_change'] = df[f'{lc_type}_ratio'] - df[f'{lc_type}_ratio_ly']
            return df

        # 计算冲突区统计
        self.df_conflict_zonal = self.df_conflict\
            .pipe(calculate_stats, "conflict_")\
            .pipe(calculate_ratios)\
            .pipe(calculate_changes)\
            .assign(c=1)

        # 计算非冲突区统计
        self.df_non_conflict_zonal = self.df_non_conflict\
            .pipe(calculate_stats, "non_conflict_")\
            .pipe(calculate_ratios)\
            .pipe(calculate_changes)\
            .assign(c=0)
        
        # 合并冲突区和非冲突区数据
        self.df_zonal_combined = pd.concat([self.df_conflict_zonal, self.df_non_conflict_zonal], ignore_index=True)
        self.df_zonal_combined.to_csv(path_data / "sample_data/extract_data" / f"{self.year}_zonal_combined.csv", )
    
    def _load_LA(self):
        self.df_LA = pd.read_csv(path_data / "sample_data/extract_data" / f"{self.year}_LA_metric.csv", index_col=0)\
            .assign(c=lambda x: x.conflict.map({"conflict": 1, "non_conflict": 0}))\
            .drop(columns=["conflict"])\
            .rename(columns={"currentyear": "shannon", "lastyear":"shannon_ly"})\
            .assign(shannon_change=lambda x: x.shannon - x.shannon_ly)
        self.df_zonal_combined = self.df_zonal_combined\
            .merge(self.df_LA, how="left", on=["idx", "c"],)
    
    def _seperate_zonal(self):
        self.df_zonal_heat = self.df_zonal_combined.query("event in [1, 4, 5]")
        self.df_zonal_drought = self.df_zonal_combined.query("event in [2, 4]")
        self.df_zonal_wet = self.df_zonal_combined.query("event in [3, 5]")

In [ ]:
year = 2020
radius = 20
data1year = Extract1YearData(year, radius=radius)

In [ ]:
class PSMMatcher:
    def __init__(self, df, 
                 covariates=['pop', 'dis2conflict', 'crop_ratio_ly', 'built_ratio_ly', 'dis2road', 'dis2bound'], 
                 treatment_col='c', caliper_ratio=0.25, _gp=None):
        self.df = df.dropna(subset=covariates).reset_index(drop=True)
        self.covariates = covariates
        self.treatment_col = treatment_col
        self.caliper_ratio = caliper_ratio
        self.propensity_scores = None
        self.ps_sd = None
        self.matched_df = None
        self._gp = _gp
        self.skip_num = 0
        
    def compute_propensity_score(self):
        X = self.df[self.covariates]
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        model = LogisticRegression(solver='liblinear', random_state=0)
        model.fit(X_scaled, self.df[self.treatment_col])
        self.propensity_scores = model.predict_proba(X_scaled)[:, 1]
        self.ps_sd = np.std(self.propensity_scores)
        self.df['ps'] = self.propensity_scores

    def match(self):
        if self.propensity_scores is None:
            self.compute_propensity_score()

        caliper = self.caliper_ratio * self.ps_sd

        df_treat = self.df[self.df[self.treatment_col] == 1].copy()
        df_ctrl = self.df[self.df[self.treatment_col] == 0].copy()

        nn = NearestNeighbors(n_neighbors=1, metric='euclidean')
        nn.fit(df_ctrl['ps'].values.reshape(-1, 1))

        matched_treat_idx = []
        matched_ctrl_idx = []
        # used_ctrl_idx = set()
        for idx in df_treat.index:
            ps_t = self.df.at[idx, 'ps']
            dist, neighbors = nn.kneighbors([[ps_t]], return_distance=True)
            dist = dist[0][0]
            nearest_idx = neighbors[0][0]
            ctrl_i = df_ctrl.index[nearest_idx]

            if dist > caliper:
                self.skip_num += 1
                # print(f"Skipping {idx} due to dist={dist:.4f} > caliper={caliper:.4f}")
                print(f"{self._gp} skip {self.skip_num}" + " "*10, end="\r")
                continue
            # if ctrl_i in used_ctrl_idx:
            #     continue

            matched_treat_idx.append(idx)
            matched_ctrl_idx.append(ctrl_i)
            # used_ctrl_idx.add(ctrl_i)

        df_treat_matched = self.df.loc[matched_treat_idx].reset_index(drop=True)
        df_ctrl_matched = self.df.loc[matched_ctrl_idx].reset_index(drop=True)

        self.matched_df = df_treat_matched.join(df_ctrl_matched, rsuffix="_n")
        print(f"{self._gp} skip {self.skip_num}" + " "*10)
        
        return self.matched_df

    def get_matched_data(self):
        return self.matched_df


In [ ]:
data_lst = []
for year in range(2002, 2024):
    data1year = Extract1YearData(year, radius=radius)
    data_lst.append(data1year)

In [ ]:
data_merge = pd.concat([data1year.df_zonal_combined for data1year in data_lst], ignore_index=True).assign(regi_short=lambda _df: _df["regi_pnas"].map(dic_region))
data_drought = pd.concat([data1year.df_zonal_drought for data1year in data_lst], ignore_index=True).assign(regi_short=lambda _df: _df["regi_pnas"].map(dic_region))
data_heat = pd.concat([data1year.df_zonal_heat for data1year in data_lst], ignore_index=True).assign(regi_short=lambda _df: _df["regi_pnas"].map(dic_region))
data_wet = pd.concat([data1year.df_zonal_wet for data1year in data_lst], ignore_index=True).assign(regi_short=lambda _df: _df["regi_pnas"].map(dic_region))

# T test

In [ ]:
def ttest_for_event(df_event, gp_col="koppen", min_size=50):
    gp_for_event = df_event.query("c == 1").groupby(gp_col, as_index=False).size()\
        .query("size > @min_size")\
        [gp_col].values
    df_event = df_event.query(f"{gp_col} in @gp_for_event")
    
    ttest_results = []
    
    for _gp in gp_for_event:
        df_event_gp = df_event.query(f"{gp_col} == @_gp")
        col_logistic = ['pop', 'dis2conflict', 'crop_ratio_ly', 'built_ratio_ly', 'dis2road', 'dis2bound']
        df_event_gp = df_event_gp.dropna(subset=col_logistic)    
        try:
            matcher = PSMMatcher(df_event_gp, _gp=_gp)
            _df_matched = matcher.match()
            # _df_matched = match_conflict_to_non_con(df_event_gp)
        except:
            print(f"Skipping group {_gp} due to insufficient data for matching.")
            continue
        
        # luc_lst = ["fire_area", "crop_change", "built_change", "forest_change", "shrubland_change", "grass_change", "wetland_change", "barren_change"]
        luc_lst = ["fire_area", "forest_change",  "wetland_change", "shannon_change"]
        for luc_ in luc_lst:
            t_stat, t_p = ttest_rel(_df_matched[f"{luc_}"].values, _df_matched[f"{luc_}_n"].values)
            ttest_results.append([_gp, luc_, t_stat, t_p])
    ttest_results = pd.DataFrame(ttest_results, columns=[gp_col, "luc", "t_stat", "t_p"])
    return ttest_results
        

In [ ]:
def match_conflict_to_non_con(df_event_gp, caliper_ratio=0.25):

    df_sample = df_event_gp.reset_index(drop=True)    
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(df_sample[['pop', 'dis2conflict', 'crop_ratio_ly', 'built_ratio_ly', 'dis2road', 'dis2bound']])
    logistic_model = LogisticRegression(solver='liblinear', random_state=0)
    logistic_model.fit(X_scaled, df_sample['c'])
    propensity_scores = logistic_model.predict_proba(X_scaled)[:, 1]

    ps_sd = np.std(propensity_scores)
    caliper = caliper_ratio * ps_sd

    df_sample = df_sample.assign(ps=propensity_scores)
    df_c_match = df_sample.query('c == 1')
    df_nc_match = df_sample.query('c == 0')

    nn = NearestNeighbors(n_neighbors=1, metric='euclidean')
    nn.fit(propensity_scores[df_sample['c'] == 0].reshape(-1, 1)) 

    # matched_indices = []
    matched_treat = []
    matched_control = []

    for idx in df_c_match.index:
        treated_score = propensity_scores[idx]
        # nearest_idx = nn.kneighbors([[treated_score]])[1][0][0]
        dist, neighbor_idx = nn.kneighbors([[treated_score]], return_distance=True)
        dist = dist[0][0]
        nearest_idx = neighbor_idx[0][0]
        if dist > caliper:
            print(f"Skipping index {idx} due to distance {dist} exceeding caliper {caliper}.")
            continue
        ctrl_i = df_nc_match.index[nearest_idx]
        matched_treat.append(idx)
        matched_control.append(ctrl_i)

    # matched_df = df_sample.loc[matched_control]
    # df_matched = df_c_match.reset_index(drop=True).join(matched_df.reset_index(drop=True), rsuffix="_n")

    df_treat = df_sample.loc[matched_treat].reset_index(drop=True)
    df_ctrl = df_sample.loc[matched_control].reset_index(drop=True)
    df_matched = df_treat.join(df_ctrl, rsuffix="_n")
    return df_matched

In [ ]:
def plot_ttest(ttest_results_drought, title="t-test for drought Event"):
    gp_col = ttest_results_drought.columns[0]
    df_t_stat = ttest_results_drought\
        .pivot(index="luc", columns=gp_col, values="t_stat")
    df_t_p = ttest_results_drought\
        .pivot(index="luc", columns=gp_col, values="t_p")
        
    f, ax = plt.subplots(figsize=(10, 6), gridspec_kw={"left": 0.15, "right": 0.95, "top": 0.9, "bottom": 0.2})
    sns.heatmap(
        df_t_stat, annot=True, fmt=".1f", cmap="coolwarm", vmax=20, vmin=-20, center=0, cbar_kws={"label": "t-statistic"}, ax=ax)

    data = df_t_stat.values
    data_p = df_t_p.values

    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            if data_p[i, j] < 0.05:
                ax.add_patch(Rectangle((j, i), 1, 1, fill=False, hatch="//", edgecolor="k", lw=0.5))
    ax.set_title(title)
    _ = plt.xticks(rotation=45, ha="right")
    return f, ax

## country

In [ ]:
filter_str = '(luc == "fire_area" & t_stat > 0) | (luc == "shannon_change" & t_stat < 0) | (luc == "wetland_change" & t_stat < 0) | (luc == "forest_change" & t_stat < 0)'

In [ ]:
ttest_results_drought = ttest_for_event(data_drought, gp_col="name_long", min_size=50)

In [ ]:
f, ax = plot_ttest(
    ttest_results_drought,#.query(filter_str),
    title="t-test for drought Event")

ax.grid(False)

In [ ]:
ttest_results_drought_filter = ttest_results_drought.query(filter_str)\
    .assign(event="drought")

In [ ]:
ttest_results_wet = ttest_for_event(data_wet, gp_col="name_long", min_size=100)

In [ ]:
f, ax = plot_ttest(ttest_results_wet, title="t-test for wet Event")

In [ ]:
ttest_results_wet_filter = ttest_results_wet.query(filter_str)\
    .assign(event="wet")

In [ ]:
ttest_results_heat = ttest_for_event(data_heat, gp_col="name_long", min_size=100)


In [ ]:
f, ax = plot_ttest(ttest_results_heat, title="t-test for heat Event")

In [ ]:
ttest_results_heat_filter = ttest_results_heat.query(filter_str)\
    .assign(event="heat")

In [ ]:
ttest_results_merge_filter = pd.concat(
    [ttest_results_drought_filter, ttest_results_wet_filter, ttest_results_heat_filter],
    ignore_index=True
)\
    .query("(event != 'wet' | luc != 'fire_area')")

gp_col = ttest_results_merge_filter.columns[0]
df_t_stat = ttest_results_merge_filter\
    .pivot(index=["event", "luc"], columns=gp_col, values="t_stat")

df_t_p = ttest_results_merge_filter\
    .pivot(index=["event", "luc"], columns=gp_col, values="t_p")